# `ptof_obs_bronze_projection`

## What this notebook does
This is the **entry point of the ISH agent observability pipeline**. It projects the two raw,
externally-owned "oil layer" audit tables into two durable, append-only `oil_obs` VIEWs
(`v_llm_bronze`, `v_ish_bronze`), adding derived flags that every downstream detector relies on so
those detectors don't each re-implement the same classification logic.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `01_bronze_projections` -- runs first, before anything else.
- **Upstream:** nothing in this repo. Reads directly from the oil-layer base tables, which are
  owned and written by systems outside this observability pipeline.
- **Downstream:** every other notebook in this repo reads `v_llm_bronze` and/or `v_ish_bronze`
  instead of the oil tables directly -- `ptof_obs_latency_detection`, `ptof_obs_mal_output`,
  `ptof_obs_hallucination_detection`, `ptof_obs_behavioral_correlation`, `ptof_obs_nightly_baseline`,
  and (via those detectors' findings tables) eventually `ptof_obs_alert`.

## Why a projection layer exists at all
Putting the derived flags (`is_blank_output`, `error_class`, `is_credential_fastfail`, etc.) here
instead of in each detector means: (1) every detector agrees on what "blank output" or "a timeout"
means, since they all read the same boolean/enum instead of re-deriving it from raw text, and
(2) if a classification rule needs to change, it changes in one place.

## Tables/views touched
- **Reads:** `mq_gmdf_dev.oil.ptof_primary__ai_llm_audit_log` (raw LLM call log -- one row per LLM
  call, owned by the agent runtime, not this repo) and `mq_gmdf_dev.oil.ptof_ish_audit`
  (raw ISH change-audit log -- one row per create/update/delete against ISH entities).
- **Writes:** `mq_gmdf_dev.oil_obs.v_llm_bronze` and `mq_gmdf_dev.oil_obs.v_ish_bronze` -- both
  `CREATE OR REPLACE VIEW`s (not materialized tables), so they always reflect the current oil-layer
  contents. Being views over append-only base tables is what makes them safe for the "Where to
  look" backtracking queries in `ptof_obs_alert.ipynb` to point at -- unlike each detector's own
  findings tables, which are `CREATE OR REPLACE`'d over a rolling window every run and go stale
  within hours.


In [0]:
%sql
-- v_llm_bronze: the single durable, queryable source of truth for every LLM call the agent made.
-- Every detector notebook (latency, malformed-output, hallucination, behavioral correlation) reads
-- this view instead of the raw oil table, so a classification rule only needs to change here.
-- CREATE OR REPLACE VIEW (not a table) -- always reflects the current contents of the base table,
-- and is safe to backtrack a Teams incident to days later since the base table is append-only.
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_llm_bronze AS
SELECT
    -- pass-through columns straight from the raw audit log: call identity, shift/batch context,
    -- which capability/model/transport handled the call, whether it succeeded, and the prompt/
    -- response payloads themselves.
    id, shift_date, shift_type, batch_nbr, capability, scheduler_run,
    model_config, transport, success, error_msg, latency_ms, called_at, ingestion_ts,

    -- write lag: seconds between when the call happened (called_at) and when it landed in this
    -- table (ingestion_ts). A growing lag is itself an observability signal (pipeline backpressure)
    -- -- consumed by ptof_obs_latency_detection's write_lag_daily output.
    unix_timestamp(ingestion_ts) - unix_timestamp(called_at) AS write_lag_s,

    user_prompt, system_prompt, response_raw, response_parsed,

    -- is_credential_fastfail: flags the specific known failure signature of a misconfigured demo
    -- model config hitting "Unable to locate credentials" immediately (fails fast, not a timeout).
    -- Surfaced by ptof_obs_latency_detection as credential_fastfail_daily so a spike in these is
    -- diagnosed as a config problem, not treated as a generic latency/error anomaly.
        CASE
      WHEN model_config = 'demo-claude-sonnet-4-6-pwc-omi'
       AND error_msg    = 'Unable to locate credentials'
        THEN true ELSE false
    END AS is_credential_fastfail,

    -- is_blank_output: the call succeeded transport-wise but returned nothing usable -- empty/
    -- null response, or an empty JSON container. This is the core signal for the blank_output
    -- detector in ptof_obs_mal_output.ipynb, which is wired into ptof_obs_alert as CRITICAL: a
    -- silent empty response is worse than a loud failure because nothing downstream complains.
            CASE
      WHEN success = true
       AND (response_parsed IS NULL
            OR length(trim(cast(response_parsed AS STRING))) = 0
            OR cast(response_parsed AS STRING) IN ('{}', '[]', 'null')
            -- valid JSON with a null how_we_ran: the one field summary's system prompt calls
            -- "always present". This shape read as healthy before.
            OR (capability = 'summary'
                AND coalesce(trim(try_parse_json(cast(response_parsed AS STRING)):how_we_ran::string), '') = ''))
        THEN true ELSE false
    END AS is_blank_output,

        -- error_class: buckets every failure into a taxonomy so downstream detectors can treat
        -- categories differently -- e.g. timeouts feed latency/anomaly detection (they're a speed
        -- problem), while auth/rate_limit/upstream_5xx feed error-rate detection (they're an
        -- external-dependency problem), and this separation is what lets
        -- capability_error_rate_sustained and latency_anomaly stay independent signals instead of
        -- double-counting the same failures.
        CASE
      WHEN success = true                                               THEN NULL
      WHEN error_msg ILIKE '%timed out%' OR error_msg ILIKE '%timeout%'  THEN 'timeout'
      WHEN error_msg ILIKE '%credential%'
        OR error_msg ILIKE '%AccessDenied%'
        OR error_msg ILIKE '%403%'
        OR error_msg ILIKE '%Forbidden%'
        OR error_msg ILIKE '%401%'
        OR error_msg ILIKE '%Unauthorized%'                             THEN 'auth'
      WHEN error_msg ILIKE '%Internal Server Error%'
        OR error_msg RLIKE '(?i)cortex\\s+5[0-9]{2}'                    THEN 'upstream_5xx'
      WHEN error_msg ILIKE '%rate limit%' OR error_msg ILIKE '%429%'     THEN 'rate_limit'
      WHEN error_msg ILIKE '%connection%'
        OR error_msg ILIKE '%WinError%'
        OR error_msg ILIKE '%reset by peer%'
        OR error_msg ILIKE '%broken pipe%'
        OR error_msg ILIKE '%ECONNRESET%'                               THEN 'connection'
      ELSE 'other'
    END AS error_class,

    -- payload sizes: leading indicator for timeout risk -- large prompts correlate with the
    -- dsa_optimize timeouts documented in ptof_obs_alert's header (a 37,006-char user prompt was
    -- the trigger case). Lets latency detection reason about "is this call slow because it's a
    -- big prompt" vs. "is this call slow for no obvious reason."
    length(cast(system_prompt   AS STRING)) AS system_prompt_chars,
    length(cast(user_prompt     AS STRING)) AS user_prompt_chars,
    length(cast(response_parsed AS STRING)) AS response_chars,

    -- resp_v: response_parsed pre-parsed as a Databricks variant/JSON value, so every downstream
    -- query that needs to reach into the response body (e.g. schema-drift field checks in
    -- ptof_obs_mal_output) can use `:field` path syntax instead of re-parsing raw text each time.
    try_parse_json(cast(response_parsed AS STRING)) AS resp_v
FROM mq_gmdf_dev.oil.ptof_primary__ai_llm_audit_log;

In [0]:
%sql
-- v_ish_bronze: durable, append-only view over the ISH change-audit log. Feeds
-- ptof_obs_behavioral_correlation (handover delivery, rapid human correction) -- the only
-- notebook that correlates LLM behavior against human/ISH-system activity.
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_ish_bronze AS
WITH j AS (
  -- j: parse both JSON blobs (the entity's state after and before the change) up front, once,
  -- so every downstream expression can use `:field` path syntax instead of re-parsing per column.
  SELECT
      id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
      try_parse_json(after_json)  AS after_v,
      try_parse_json(before_json) AS before_v
  FROM mq_gmdf_dev.oil.ptof_ish_audit
)
SELECT
    id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
    before_v, after_v,

    -- shift_date_norm / shift_type_norm / batch_id_norm: ISH entities have historically used
    -- inconsistent field names for the same concept (shift_date vs shift_date_key, etc.) --
    -- these normalize to one name so behavioral correlation can join ISH activity to LLM calls
    -- (which use shift_date/shift_type/batch_nbr in v_llm_bronze) without per-caller guesswork.
    CAST(coalesce(
        after_v:shift_date::string,
        after_v:shift_date_key::string
    ) AS DATE) AS shift_date_norm,
    coalesce(
        after_v:shift_type::string,
        after_v:shift_label::string
    ) AS shift_type_norm,
    coalesce(
        after_v:batch_id::string,
        after_v:content.batch_id::string
    ) AS batch_id_norm,

    -- is_email_disabled_gate: flags the specific known case where a handover email was
    -- deliberately not sent because the EMAIL_ENABLED feature flag was off. Lets
    -- handover_delivery_rate distinguish "the system chose not to send" from "the system tried
    -- and failed to send" -- the latter is the CRITICAL failure mode, the former is expected.
    CASE
      WHEN entity_type = 'HandoverEmail'
       AND after_v:sent::boolean = false
       AND after_v:reason::string LIKE '%EMAIL_ENABLED=false%'
      THEN true ELSE false
    END AS is_email_disabled_gate
FROM j;